In [1]:
import pandas as pd
from pathlib import Path

data_path = Path("../data/processed/eda_data.csv")

df = pd.read_csv(data_path)
print(df.shape)
df.head()

(113425, 31)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,order_month
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,4.0,500.0,19.0,8.0,13.0,housewares,9350.0,maua,SP,2017-10
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,1.0,400.0,19.0,13.0,19.0,perfumery,31570.0,belo horizonte,SP,2018-07
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,1.0,420.0,24.0,19.0,21.0,auto,14840.0,guariba,SP,2018-08
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,3.0,450.0,30.0,10.0,20.0,pet_shop,31842.0,belo horizonte,MG,2017-11
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,4.0,250.0,51.0,15.0,15.0,stationery,8752.0,mogi das cruzes,SP,2018-02


In [2]:
customer_features = (
    df.groupby("customer_unique_id")
      .agg(
          order_count=("order_id", "nunique"),
          total_spend=("price", "sum"),
          average_item_price=("price", "mean")
      )
      .reset_index()
)
print(customer_features.shape)
print(customer_features.head())

(96096, 4)
                 customer_unique_id  order_count  total_spend  \
0  0000366f3b9a7992bf8c76cfdf3221e2            1       129.90   
1  0000b849f77a49e4a4ce2b2a4ca5be3f            1        18.90   
2  0000f46a3911fa3c0805444483337064            1        69.00   
3  0000f6ccb0745a6a4b88665a16c9f078            1        25.99   
4  0004aac84e0df4da2b147fca70cf8255            1       180.00   

   average_item_price  
0              129.90  
1               18.90  
2               69.00  
3               25.99  
4              180.00  


In [3]:
customer_features["average_order_value"] = (
    customer_features["total_spend"] /
    customer_features["order_count"]
)

In [4]:
customer_features.columns

Index(['customer_unique_id', 'order_count', 'total_spend',
       'average_item_price', 'average_order_value'],
      dtype='object')

In [5]:
print(customer_features.head())

                 customer_unique_id  order_count  total_spend  \
0  0000366f3b9a7992bf8c76cfdf3221e2            1       129.90   
1  0000b849f77a49e4a4ce2b2a4ca5be3f            1        18.90   
2  0000f46a3911fa3c0805444483337064            1        69.00   
3  0000f6ccb0745a6a4b88665a16c9f078            1        25.99   
4  0004aac84e0df4da2b147fca70cf8255            1       180.00   

   average_item_price  average_order_value  
0              129.90               129.90  
1               18.90                18.90  
2               69.00                69.00  
3               25.99                25.99  
4              180.00               180.00  


In [6]:
df["order_purchase_timestamp"].max()

'2018-10-17 17:30:18'

In [7]:
last_order_date = (
    df.groupby("customer_unique_id")["order_purchase_timestamp"]
      .max()
      .reset_index()
)

In [8]:
print(last_order_date.head())
print(last_order_date.shape)

                 customer_unique_id order_purchase_timestamp
0  0000366f3b9a7992bf8c76cfdf3221e2      2018-05-10 10:56:27
1  0000b849f77a49e4a4ce2b2a4ca5be3f      2018-05-07 11:11:27
2  0000f46a3911fa3c0805444483337064      2017-03-10 21:05:03
3  0000f6ccb0745a6a4b88665a16c9f078      2017-10-12 20:29:41
4  0004aac84e0df4da2b147fca70cf8255      2017-11-14 19:45:42
(96096, 2)


In [9]:
df["order_purchase_timestamp"] = pd.to_datetime(
    df["order_purchase_timestamp"]
)

last_order_date["order_purchase_timestamp"] = pd.to_datetime(
    last_order_date["order_purchase_timestamp"]
)

In [10]:
reference_date = df["order_purchase_timestamp"].max()

last_order_date["days_since_last_order"] = (
    reference_date - last_order_date["order_purchase_timestamp"]
).dt.days

print(last_order_date.head())

                 customer_unique_id order_purchase_timestamp  \
0  0000366f3b9a7992bf8c76cfdf3221e2      2018-05-10 10:56:27   
1  0000b849f77a49e4a4ce2b2a4ca5be3f      2018-05-07 11:11:27   
2  0000f46a3911fa3c0805444483337064      2017-03-10 21:05:03   
3  0000f6ccb0745a6a4b88665a16c9f078      2017-10-12 20:29:41   
4  0004aac84e0df4da2b147fca70cf8255      2017-11-14 19:45:42   

   days_since_last_order  
0                    160  
1                    163  
2                    585  
3                    369  
4                    336  


In [11]:
customer_features = customer_features.merge(
    last_order_date[
        ["customer_unique_id", "days_since_last_order"]
    ],
    on="customer_unique_id",
    how="left",
    validate="one_to_one"
)

print(customer_features.head())
print(customer_features.shape)

                 customer_unique_id  order_count  total_spend  \
0  0000366f3b9a7992bf8c76cfdf3221e2            1       129.90   
1  0000b849f77a49e4a4ce2b2a4ca5be3f            1        18.90   
2  0000f46a3911fa3c0805444483337064            1        69.00   
3  0000f6ccb0745a6a4b88665a16c9f078            1        25.99   
4  0004aac84e0df4da2b147fca70cf8255            1       180.00   

   average_item_price  average_order_value  days_since_last_order  
0              129.90               129.90                    160  
1               18.90                18.90                    163  
2               69.00                69.00                    585  
3               25.99                25.99                    369  
4              180.00               180.00                    336  
(96096, 6)


In [12]:
customer_features.describe()


,order_count,total_spend,average_item_price,average_order_value,days_since_last_order
count,96096.000000,96096.000000,95420.000000,96096.000000,96096.000000
mean,1.034809,141.438184,126.516509,137.163463,287.735691
std,0.214384,217.215904,191.743686,210.735847,153.414676
min,1.000000,0.000000,0.850000,0.000000,0.000000
25%,1.000000,45.990000,42.900000,45.827500,163.000000
50%,1.000000,89.000000,79.080000,85.990000,268.000000
75%,1.000000,154.000000,139.900000,149.900000,397.000000
max,17.000000,13440.000000,6735.000000,13440.000000,772.000000


In [13]:
customer_features[
    customer_features["average_item_price"].isna()
].head()

,customer_unique_id,order_count,total_spend,average_item_price,average_order_value,days_since_last_order
123,004ad212c650c96c8a48f30855a7bcb8,1,0.0,NaN,0.0,388
177,0071f46a072a9ae25bbe4438b15efe9c,1,0.0,NaN,0.0,623
238,009b0127b727ab0ba422f6d9604487c7,1,0.0,NaN,0.0,764
372,00f0b70fdcb8a6e1671b52a2472bd41f,1,0.0,NaN,0.0,742
475,013b083525868a0437652ebf071cda03,1,0.0,NaN,0.0,318


In [14]:
customer_features[
    customer_features["average_item_price"].isna()
].shape

(676, 6)

In [15]:
customer_location = (
    df.groupby("customer_unique_id")
      .agg(
          customer_state=("customer_state", "first"),
          customer_city=("customer_city", "first")
      )
      .reset_index()
)

print(customer_location.head())
print(customer_location.shape)

                 customer_unique_id customer_state customer_city
0  0000366f3b9a7992bf8c76cfdf3221e2             SP       cajamar
1  0000b849f77a49e4a4ce2b2a4ca5be3f             SP        osasco
2  0000f46a3911fa3c0805444483337064             SC      sao jose
3  0000f6ccb0745a6a4b88665a16c9f078             PA         belem
4  0004aac84e0df4da2b147fca70cf8255             SP      sorocaba
(96096, 3)


In [16]:
state_counts = (
    df.groupby("customer_unique_id")["customer_state"]
      .nunique()
)

print(state_counts.value_counts())

customer_state
1    96057
2       38
3        1
Name: count, dtype: int64


In [17]:
customer_features = customer_features.merge(
    customer_location,
    on="customer_unique_id",
    how="left",
    validate="one_to_one"
)

print(customer_features.head())
print(customer_features.shape)

                 customer_unique_id  order_count  total_spend  \
0  0000366f3b9a7992bf8c76cfdf3221e2            1       129.90   
1  0000b849f77a49e4a4ce2b2a4ca5be3f            1        18.90   
2  0000f46a3911fa3c0805444483337064            1        69.00   
3  0000f6ccb0745a6a4b88665a16c9f078            1        25.99   
4  0004aac84e0df4da2b147fca70cf8255            1       180.00   

   average_item_price  average_order_value  days_since_last_order  \
0              129.90               129.90                    160   
1               18.90                18.90                    163   
2               69.00                69.00                    585   
3               25.99                25.99                    369   
4              180.00               180.00                    336   

  customer_state customer_city  
0             SP       cajamar  
1             SP        osasco  
2             SC      sao jose  
3             PA         belem  
4             SP      sorocab

In [18]:
customer_features = customer_features.drop(columns=["customer_city"])

print(customer_features.shape)
print(customer_features.head())

(96096, 7)
                 customer_unique_id  order_count  total_spend  \
0  0000366f3b9a7992bf8c76cfdf3221e2            1       129.90   
1  0000b849f77a49e4a4ce2b2a4ca5be3f            1        18.90   
2  0000f46a3911fa3c0805444483337064            1        69.00   
3  0000f6ccb0745a6a4b88665a16c9f078            1        25.99   
4  0004aac84e0df4da2b147fca70cf8255            1       180.00   

   average_item_price  average_order_value  days_since_last_order  \
0              129.90               129.90                    160   
1               18.90                18.90                    163   
2               69.00                69.00                    585   
3               25.99                25.99                    369   
4              180.00               180.00                    336   

  customer_state  
0             SP  
1             SP  
2             SC  
3             PA  
4             SP  


In [21]:
customer_features.columns

Index(['customer_unique_id', 'order_count', 'total_spend',
       'average_item_price', 'average_order_value', 'days_since_last_order',
       'customer_state'],
      dtype='object')

In [22]:
first_order_date = (
    df.groupby("customer_unique_id")["order_purchase_timestamp"]
      .min()
      .reset_index()
)

first_order_date["order_purchase_timestamp"] = pd.to_datetime(
    first_order_date["order_purchase_timestamp"]
)

first_order_date.head()

,customer_unique_id,order_purchase_timestamp
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42


In [23]:
first_order_date["customer_lifetime_days"] = (
    last_order_date["order_purchase_timestamp"]
    - first_order_date["order_purchase_timestamp"]
).dt.days

first_order_date.head()

,customer_unique_id,order_purchase_timestamp,customer_lifetime_days
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27,0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27,0
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03,0
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41,0
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42,0


In [24]:
customer_features = customer_features.merge(
    first_order_date[
        ["customer_unique_id", "customer_lifetime_days"]
    ],
    on="customer_unique_id",
    how="left",
    validate="one_to_one"
)

print(customer_features.shape)
print(customer_features.head())

(96096, 8)
                 customer_unique_id  order_count  total_spend  \
0  0000366f3b9a7992bf8c76cfdf3221e2            1       129.90   
1  0000b849f77a49e4a4ce2b2a4ca5be3f            1        18.90   
2  0000f46a3911fa3c0805444483337064            1        69.00   
3  0000f6ccb0745a6a4b88665a16c9f078            1        25.99   
4  0004aac84e0df4da2b147fca70cf8255            1       180.00   

   average_item_price  average_order_value  days_since_last_order  \
0              129.90               129.90                    160   
1               18.90                18.90                    163   
2               69.00                69.00                    585   
3               25.99                25.99                    369   
4              180.00               180.00                    336   

  customer_state  customer_lifetime_days  
0             SP                       0  
1             SP                       0  
2             SC                       0  
3          

In [25]:
customer_features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96096 entries, 0 to 96095
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   customer_unique_id      96096 non-null  object 
 1   order_count             96096 non-null  int64  
 2   total_spend             96096 non-null  float64
 3   average_item_price      95420 non-null  float64
 4   average_order_value     96096 non-null  float64
 5   days_since_last_order   96096 non-null  int64  
 6   customer_state          96096 non-null  object 
 7   customer_lifetime_days  96096 non-null  int64  
dtypes: float64(3), int64(3), object(2)
memory usage: 5.9+ MB


In [26]:
customer_features.describe()

,order_count,total_spend,average_item_price,average_order_value,days_since_last_order,customer_lifetime_days
count,96096.000000,96096.000000,95420.000000,96096.000000,96096.000000,96096.000000
mean,1.034809,141.438184,126.516509,137.163463,287.735691,2.711507
std,0.214384,217.215904,191.743686,210.735847,153.414676,25.393343
min,1.000000,0.000000,0.850000,0.000000,0.000000,0.000000
25%,1.000000,45.990000,42.900000,45.827500,163.000000,0.000000
50%,1.000000,89.000000,79.080000,85.990000,268.000000,0.000000
75%,1.000000,154.000000,139.900000,149.900000,397.000000,0.000000
max,17.000000,13440.000000,6735.000000,13440.000000,772.000000,633.000000


In [27]:
(customer_features["customer_lifetime_days"] > 0).sum()

2122

In [28]:
customer_features["customer_lifetime_days"].describe(
    percentiles=[0.90, 0.95, 0.99]
)

count    96096.000000
mean         2.711507
std         25.393343
min          0.000000
50%          0.000000
90%          0.000000
95%          0.000000
99%         96.000000
max        633.000000
Name: customer_lifetime_days, dtype: float64

In [ ]:
customer_features["customer_lifetime_days"].describe(
    percentiles=[0.90, 0.95, 0.99]
)

count    96096.000000
mean         2.711507
std         25.393343
min          0.000000
50%          0.000000
90%          0.000000
95%          0.000000
99%         96.000000
max        633.000000
Name: customer_lifetime_days, dtype: float64

## 1. Purchase Frequency

In [ ]:
# customer_lifetime_days = days between first and last order
# for one-time customers this is 0, so we clip to min 1 to avoid division by zero
# frequency = orders per 30-day period over their lifetime

order_dates = (
    df.groupby("customer_unique_id")["order_purchase_timestamp"]
      .agg(first_order="min", last_order="max")
      .reset_index()
)
order_dates["customer_lifetime_days"] = (
    order_dates["last_order"] - order_dates["first_order"]
).dt.days

order_dates = order_dates.merge(
    customer_features[["customer_unique_id", "order_count"]],
    on="customer_unique_id"
)
order_dates["purchase_frequency"] = (
    order_dates["order_count"] / order_dates["customer_lifetime_days"].clip(lower=1) * 30
)

customer_features = customer_features.merge(
    order_dates[["customer_unique_id", "purchase_frequency"]],
    on="customer_unique_id",
    how="left",
    validate="one_to_one"
)
customer_features["purchase_frequency"].describe()

## 2. Average Days Between Orders

In [ ]:
# only meaningful for customers with 2+ orders
# one-time customers get NaN - using 0 would be misleading

def avg_days_between(timestamps):
    sorted_ts = sorted(timestamps)
    if len(sorted_ts) < 2:
        return float('nan')
    diffs = [(sorted_ts[i+1] - sorted_ts[i]).days for i in range(len(sorted_ts)-1)]
    return sum(diffs) / len(diffs)

avg_days = (
    df.groupby("customer_unique_id")["order_purchase_timestamp"]
      .apply(avg_days_between)
      .reset_index()
      .rename(columns={"order_purchase_timestamp": "avg_days_between_orders"})
)

customer_features = customer_features.merge(
    avg_days, on="customer_unique_id", how="left", validate="one_to_one"
)
# NaN count = one-time customers (expected)
print("NaN count:", customer_features["avg_days_between_orders"].isna().sum())
customer_features["avg_days_between_orders"].describe()

## 3. Unique Categories and Preferred Category

In [ ]:
# unique_categories: distinct product categories per customer
# preferred_category: most frequently purchased category, ties broken by first alphabetically

cat_features = (
    df.groupby("customer_unique_id")["product_category_name_english"]
      .agg(
          unique_categories="nunique",
          preferred_category=lambda x: x.mode().iloc[0] if not x.mode().empty else float('nan')
      )
      .reset_index()
)

customer_features = customer_features.merge(
    cat_features, on="customer_unique_id", how="left", validate="one_to_one"
)
print(customer_features[["unique_categories", "preferred_category"]].head())

## 4. Average Review Score

In [ ]:
# df is item-level: one order can have multiple rows
# reviews are per order - aggregate at order level first to avoid double counting

reviews_raw = pd.read_csv("../data/processed/reviews.csv")[["order_id", "review_score"]]
order_review = reviews_raw.groupby("order_id")["review_score"].mean().reset_index()

order_customer = df[["order_id", "customer_unique_id"]].drop_duplicates(subset="order_id")
order_review = order_review.merge(order_customer, on="order_id", how="left")

avg_review = (
    order_review.groupby("customer_unique_id")["review_score"]
               .mean()
               .reset_index()
               .rename(columns={"review_score": "avg_review_score"})
)

customer_features = customer_features.merge(
    avg_review, on="customer_unique_id", how="left", validate="one_to_one"
)
# NaN = customer had no linked review - kept as NaN, not imputed
print("NaN count:", customer_features["avg_review_score"].isna().sum())
customer_features["avg_review_score"].describe()

## 5. Late Delivery Experience

In [ ]:
# late_order_rate = proportion of delivered orders that arrived after estimated date
# aggregate at order level to avoid counting multi-item orders multiple times
# only use rows where both delivery dates exist

orders_raw = pd.read_csv("../data/processed/orders.csv", parse_dates=[
    "order_delivered_customer_date", "order_estimated_delivery_date"
])

delivered = orders_raw[
    (orders_raw["order_status"] == "delivered") &
    orders_raw["order_delivered_customer_date"].notna() &
    orders_raw["order_estimated_delivery_date"].notna()
].copy()

delivered["is_late"] = (
    delivered["order_delivered_customer_date"] > delivered["order_estimated_delivery_date"]
).astype(int)

delivered = delivered.merge(
    df[["order_id", "customer_unique_id"]].drop_duplicates(subset="order_id"),
    on="order_id", how="left"
)

late_rate = (
    delivered.groupby("customer_unique_id")["is_late"]
             .mean()
             .reset_index()
             .rename(columns={"is_late": "late_order_rate"})
)

customer_features = customer_features.merge(
    late_rate, on="customer_unique_id", how="left", validate="one_to_one"
)
# NaN = customer had no delivered orders with both dates present
print("NaN count:", customer_features["late_order_rate"].isna().sum())
customer_features["late_order_rate"].describe()

## Final Summary

In [ ]:
print("Shape:", customer_features.shape)
print("\nColumns:", customer_features.columns.tolist())
print("\nMissing values:\n", customer_features.isna().sum())
print("\nDescriptive stats:")
customer_features.describe()

In [ ]:
customer_features.to_csv("../data/processed/customer_features.csv", index=False)
print("Saved: customer_features.csv", customer_features.shape)